In [1]:
# Install Required Libraries
!pip install pandas scikit-learn numpy

In [15]:
import pandas as pd
import numpy as np

pd.options.mode.chained_assignment = None

df = pd.read_excel('/content/drive/MyDrive/DATA FOR COLLAB/intent_classification_dataset (1).xlsx')

df.head()

,id,customer_text,label,sentiment
0,1,Your effort on duplicate charge is commendable. 😡,Feedback,Neutral
1,2,I'M SO FRUSTRATED ABOUT PASSWORD RESET.,Complaint,Negative
2,3,Can you arrange to change my billing date abou...,Request,Neutral
3,4,This is a nightmare - compatibility issue. 🤔,Complaint,Negative
4,5,COULD YOU REVERSE THE CHARGE FOR THE CANCELLAT...,Request,Neutral


In [4]:
intents = df[['customer_text', 'label']]

intents

,customer_text,label
0,Your effort on duplicate charge is commendable. 😡,Feedback
1,I'M SO FRUSTRATED ABOUT PASSWORD RESET.,Complaint
2,Can you arrange to change my billing date abou...,Request
3,This is a nightmare - compatibility issue. 🤔,Complaint
4,COULD YOU REVERSE THE CHARGE FOR THE CANCELLAT...,Request
...,...,...
49995,Can you clarify warehouse delay for me?,Inquiry
49996,Your approach to plan confusion is wonderful. 😭,Feedback
49997,Please make sure to escalate my case regarding...,Request
49998,"Dear Support Team,\n\nYou might want to recons...",Feedback


In [5]:
intents.shape

(50000, 2)

In [6]:
intents.isnull().sum()

,0
customer_text,0
label,0


In [7]:
intents.duplicated().sum()

np.int64(0)

In [8]:
intents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_text  50000 non-null  object
 1   label          50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [9]:
intents['label'].value_counts()

,count
label,
Feedback,12500
Complaint,12500
Request,12500
Inquiry,12500


In [10]:
# Model 1: TF-IDF + LinearSVC

# Customer Text -> Text Cleaning (Lowercase, Remove Space, Remove Punctuation, Remove Stopwords, Lemmatization) -> TF-IDF Vectorizer -> Train/Test Split -> LinearSVC -> Prediction

In [11]:
!pip install nltk

In [22]:
# Text preprossing

import re

intents['clean_text'] = intents['clean_text'].str.lower()

intents['clean_text']

,clean_text
0,your effort on duplicate charge is commendable. 😡
1,i'm so frustrated about password reset.
2,can you arrange to change my billing date abou...
3,this is a nightmare - compatibility issue. 🤔
4,could you reverse the charge for the cancellat...
...,...
49995,can you clarify warehouse delay for me?
49996,your approach to plan confusion is wonderful. 😭
49997,please make sure to escalate my case regarding...
49998,"dear support team,\n\nyou might want to recons..."


In [23]:
# Remove spaces in text

intents['clean_text'] = intents['clean_text'].str.strip()

intents['clean_text']

,clean_text
0,your effort on duplicate charge is commendable. 😡
1,i'm so frustrated about password reset.
2,can you arrange to change my billing date abou...
3,this is a nightmare - compatibility issue. 🤔
4,could you reverse the charge for the cancellat...
...,...
49995,can you clarify warehouse delay for me?
49996,your approach to plan confusion is wonderful. 😭
49997,please make sure to escalate my case regarding...
49998,"dear support team,\n\nyou might want to recons..."


In [24]:
import string

def remove_punctuation(text):
  for punctuation in string.punctuation:
    text = text.replace(punctuation, '')
  return text

In [25]:
# Remove punctuation in texts
intents['clean_text'] = intents['clean_text'].apply(remove_punctuation)

In [26]:
import nltk
from nltk.corpus import stopwords

In [27]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [29]:
# Remove Stop Words from texts
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  text = [word for word in text.split() if word not in stop_words]
  return " ".join(text)

In [30]:
intents['clean_text'] = intents['clean_text'].apply(remove_stopwords)

intents['clean_text']

,clean_text
0,effort duplicate charge commendable 😡
1,im frustrated password reset
2,arrange change billing date cancellation problem
3,nightmare compatibility issue 🤔
4,could reverse charge cancellation problem
...,...
49995,clarify warehouse delay
49996,approach plan confusion wonderful 😭
49997,please make sure escalate case regarding plan ...
49998,dear support team might want reconsider appoin...


In [31]:
# Lemmatization
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [32]:
def lemmatize_word(text):
  words = text.split()
  words = [lemmatizer.lemmatize(word, pos="v") for word in words]
  return " ".join(words)

In [33]:
intents['clean_text'] = intents['clean_text'].apply(lemmatize_word)

intents['clean_text']

,clean_text
0,effort duplicate charge commendable 😡
1,im frustrate password reset
2,arrange change bill date cancellation problem
3,nightmare compatibility issue 🤔
4,could reverse charge cancellation problem
...,...
49995,clarify warehouse delay
49996,approach plan confusion wonderful 😭
49997,please make sure escalate case regard plan con...
49998,dear support team might want reconsider appoin...


In [34]:
# Convert Text into Numbers (TF-IDF)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

In [35]:
X = tfidf.fit_transform(intents["clean_text"]) # Feature
y = intents["label"]                           # Label

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [37]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(40000, 911)
(10000, 911)
(40000,)
(10000,)


In [38]:
from sklearn.svm import LinearSVC

svc_model = LinearSVC(dual=False)
svc_model.fit(X_train, y_train)

LinearSVC(dual=False)

In [39]:
y_pred_svc1 = svc_model.predict(X_test)

In [40]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_svc1)
print("Accuracy:", accuracy)

Accuracy: 0.9889


In [41]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_svc1))

              precision    recall  f1-score   support

   Complaint       1.00      1.00      1.00      2464
    Feedback       0.99      0.97      0.98      2569
     Inquiry       0.97      0.99      0.98      2465
     Request       1.00      1.00      1.00      2502

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [42]:
# Model 2: **Sentence-BERT + LinearSVC

# Customer Text -> Text Cleaning (only remove extra spaces if needed) -> Sentence-BERT -> 384-D Embeddings -> Train/Test Split -> LinearSVC -> Prediction

In [43]:
!pip install sentence-transformers

In [44]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [46]:
intents['clean_text2'] = intents['customer_text'].str.strip()

intents.head(3)

,customer_text,label,clean_text,clean_text2
0,Your effort on duplicate charge is commendable. 😡,Feedback,effort duplicate charge commendable 😡,Your effort on duplicate charge is commendable. 😡
1,I'M SO FRUSTRATED ABOUT PASSWORD RESET.,Complaint,im frustrate password reset,I'M SO FRUSTRATED ABOUT PASSWORD RESET.
2,Can you arrange to change my billing date abou...,Request,arrange change bill date cancellation problem,Can you arrange to change my billing date abou...


In [48]:
text = intents['clean_text2'].tolist()

embeddings = model.encode(text) # embedding of text

In [49]:
print(embeddings)

[[-0.08035507  0.03542759  0.0052787  ... -0.07308364  0.00176701
   0.05040459]
 [-0.05372667 -0.10159586 -0.00512117 ...  0.03999036 -0.05125597
  -0.05333193]
 [-0.02984883  0.03954959  0.04486804 ... -0.03644855  0.01150401
  -0.07177876]
 ...
 [ 0.02607697  0.04015997  0.06829841 ... -0.06613932 -0.05709428
   0.00535465]
 [-0.01609324  0.01531177  0.0732848  ... -0.01646505 -0.00402797
   0.00460773]
 [-0.04238547  0.0354939   0.07181546 ...  0.05662278 -0.06669249
  -0.02557115]]


In [50]:
embeddings.shape

(50000, 384)

In [51]:
X = embeddings
y = intents["label"]

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(40000, 384)
(10000, 384)
(40000,)
(10000,)


In [54]:
from sklearn.svm import LinearSVC

svm_model2 = LinearSVC()

svm_model2.fit(X_train, y_train)

LinearSVC()

In [55]:
y_pred_svm2 = svm_model2.predict(X_test)

In [57]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_svm2)

print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred_svm2))

Accuracy: 0.9987
              precision    recall  f1-score   support

   Complaint       1.00      1.00      1.00      2464
    Feedback       1.00      1.00      1.00      2569
     Inquiry       1.00      1.00      1.00      2465
     Request       1.00      1.00      1.00      2502

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [58]:
# Save model

import pickle

pickle.dump(svm_model2, open("Intentsclassification.pkl", "wb"))

print("Model saved successfully")

Model saved successfully
